In [2]:
import numpy as np

import torch
import torch.nn as nn

from captum.attr import (
    GradientShap,
    DeepLift,
    DeepLiftShap,
    IntegratedGradients,
    LayerConductance,
    NeuronConductance,
    NoiseTunnel,
)

class ToyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin1 = nn.Linear(3, 3)
        self.relu = nn.ReLU()
        self.lin2 = nn.Linear(3, 2)

        # initialize weights and biases
        self.lin1.weight = nn.Parameter(torch.arange(-4.0, 5.0).view(3, 3))
        self.lin1.bias = nn.Parameter(torch.zeros(1,3))
        self.lin2.weight = nn.Parameter(torch.arange(-3.0, 3.0).view(2, 3))
        self.lin2.bias = nn.Parameter(torch.ones(1,2))

    def forward(self, input):
        return self.lin2(self.relu(self.lin1(input)))

In [3]:
model = ToyModel()
model.eval()

ToyModel(
  (lin1): Linear(in_features=3, out_features=3, bias=True)
  (relu): ReLU()
  (lin2): Linear(in_features=3, out_features=2, bias=True)
)

In [4]:
torch.manual_seed(123)
np.random.seed(123)

In [5]:
input = torch.rand(2, 3)
baseline = torch.zeros(2, 3)

In [6]:
ig = IntegratedGradients(model)
attributions, delta = ig.attribute(input, baseline, target=0, return_convergence_delta=True)
print('IG Attributions:', attributions)
print('Convergence Delta:', delta)

IG Attributions: tensor([[-0.5922, -1.5497, -1.0067],
        [ 0.0000, -0.2219, -5.1991]], dtype=torch.float64)
Convergence Delta: tensor([ 5.9605e-08, -8.8818e-16], dtype=torch.float64)


In [7]:
gs = GradientShap(model)

# We define a distribution of baselines and draw `n_samples` from that
# distribution in order to estimate the expectations of gradients across all baselines
baseline_dist = torch.randn(10, 3) * 0.001
attributions, delta = gs.attribute(input, stdevs=0.09, n_samples=4, baselines=baseline_dist, target=0, return_convergence_delta=True)
print('GradientShap Attributions:', attributions)
print('Convergence Delta:', delta)

GradientShap Attributions: tensor([[-0.1542, -1.6229, -1.5835],
        [-0.3916, -0.2836, -4.6851]])
Convergence Delta: tensor([ 0.0000, -0.0005, -0.0029, -0.0084, -0.0087, -0.0405,  0.0000, -0.0084])


In [8]:
deltas_per_example = torch.mean(delta.reshape(input.shape[0], -1), dim=1)

In [9]:
dl = DeepLift(model)
attributions, delta = dl.attribute(input, baseline, target=0, return_convergence_delta=True)
print('DeepLift Attributions:', attributions)
print('Convergence Delta:', delta)

DeepLift Attributions: tensor([[-0.5922, -1.5497, -1.0067],
        [ 0.0000, -0.2219, -5.1991]], grad_fn=<MulBackward0>)
Convergence Delta: tensor([0., 0.])


/home/zy45/anaconda3/envs/ai/lib/python3.10/site-packages/captum/_utils/gradient.py:57: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  warnings.warn(
/home/zy45/anaconda3/envs/ai/lib/python3.10/site-packages/captum/attr/_core/deep_lift.py:304: UserWarning: Setting forward, backward hooks and attributes on non-linear
               activations. The hooks and attributes will be removed
            after the attribution is finished
  warnings.warn(


In [10]:
dl = DeepLiftShap(model)
attributions, delta = dl.attribute(input, baseline_dist, target=0, return_convergence_delta=True)
print('DeepLiftSHAP Attributions:', attributions)
print('Convergence Delta:', delta)

DeepLiftSHAP Attributions: tensor([[-5.8452e-01, -1.5447e+00, -1.0094e+00],
        [ 6.4696e-04, -2.2256e-01, -5.1892e+00]], grad_fn=<MeanBackward1>)
Convergence Delta: tensor([ 0.0000e+00, -2.3842e-07,  0.0000e+00,  2.3842e-07,  0.0000e+00,
         0.0000e+00,  0.0000e+00,  2.3842e-07,  2.3842e-07,  4.7684e-07,
         4.7684e-07,  0.0000e+00, -4.7684e-07,  0.0000e+00, -4.7684e-07,
        -4.7684e-07,  0.0000e+00, -4.7684e-07,  4.7684e-07,  0.0000e+00])


In [11]:
deltas_per_example = torch.mean(delta.reshape(input.shape[0], -1), dim=1)

In [12]:
ig = IntegratedGradients(model)
nt = NoiseTunnel(ig)
attributions, delta = nt.attribute(input, nt_type='smoothgrad', stdevs=0.02, nt_samples=4,
      baselines=baseline, target=0, return_convergence_delta=True)
print('IG + SmoothGrad Attributions:', attributions)
print('Convergence Delta:', delta)

IG + SmoothGrad Attributions: tensor([[-0.4574, -1.5493, -1.0893],
        [ 0.0000, -0.2647, -5.1619]], dtype=torch.float64)
Convergence Delta: tensor([-5.9605e-08, -5.9605e-08,  2.9802e-07, -1.7881e-07,  4.1723e-07,
        -8.8818e-16,  2.2352e-07, -8.8818e-16], dtype=torch.float64)


In [13]:
nc = NeuronConductance(model, model.lin1)
attributions = nc.attribute(input, neuron_selector=1, target=0)
print('Neuron Attributions:', attributions)

Neuron Attributions: tensor([[ 0.0000,  0.0000,  0.0000],
        [ 1.3358,  0.0000, -1.6811]])
